# 11. lecke - Ügynök-ügynök közötti (A2A) protokoll


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mi az A2A protokoll?

The **Ügynök-ügynök (A2A) protokoll** egy nyílt szabvány, amely lehetővé teszi, hogy a MI-ügynökök kommunikáljanak,
felfedezzék egymást és együttműködjenek — még akkor is, ha különböző keretrendszerekre épülnek vagy hosztolva
vannak különböző szolgáltatásoknál.

Key concepts:

- **Felfedezés** – Az ügynökök közzétesznek egy *Ügynökkártyát*, amely leírja a képességeiket, megkönnyítve
  hogy más ügynökök (vagy orchestrátorok) könnyen megtalálják a megfelelő specialistát egy feladathoz.
- **Üzenetküldés** – Az ügynökök strukturált üzeneteket cserélnek egy közös protokollon keresztül, így egy
  egy ügynöktől származó kérés érthető és teljesíthető legyen egy másik ügynök által, függetlenül annak belső
  megvalósításától.
- **Feladat életciklusa** – Az A2A olyan állapotokat határoz meg, mint *beküldött*, *feldolgozás alatt*, *befejezett* és
  *sikertelen*, ami teljes rálátást ad az orchestrátornak arra, hogyan halad egy delegált feladat.

Ebben a leckében A2A-stílusú együttműködést szimulálunk azzal, hogy három specializált utazási ügynököt
bekötünk egy munkafolyamatba, ahol minden ügynök hozzájárul szaktudásával, és továbbadja az eredményeket a következőnek.


## Szakosodott utazási ügynökök létrehozása


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Többügynökös együttműködés munkafolyamaton keresztül

A három ügynököt egy egymást követő munkafolyamatba kapcsoljuk, amely tükrözi az A2A üzenetátadást:

1. **CurrencyExchangeAgent** megkapja a felhasználó kérését, és pénzváltási útmutatást ad.
2. **ActivityPlannerAgent** megkapja a kibővített kontextust, és tevékenység-ajánlásokat ad hozzá.
3. **TravelManagerAgent** mindkét bemenetet feldolgozza, és egy végleges utazási összefoglalót készít.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Az A2A megértése éles környezetben

Éles környezetben az A2A protokoll erőteljes, szolgáltatások közötti forgatókönyveket nyit meg:

| Capability | Description |
|---|---|
| **Cross-framework interop** | Egy keretrendszerben épített ügynök feladatokat adhat át bármely más, A2A-kompatibilis keretrendszerrel készült ügynöknek, lehetővé téve a valódi szervezetek közötti együttműködést. |
| **Service boundaries** | Az ügynökök külön mikroszolgáltatásokban, felhő régiókban vagy akár különböző szervezetekben is futhatnak, miközben zökkenőmentesen együttműködnek. |
| **Dynamic discovery** | Egy orchestrator futásidőben lekérdezheti az Agent Card regisztert, hogy megtalálja a legalkalmasabb specialistát egy adott alfeladathoz. |
| **Streaming & push notifications** | Az A2A támogatja a Server-Sent Events (SSE) használatát valós idejű előrehaladási frissítésekhez, valamint push-értesítéseket a hosszú ideig tartó feladatokhoz. |

A fenti munkafolyamat, amit létrehoztunk, ennek a mintának egy leegyszerűsített, folyamaton belüli verziója. Egy valós telepítésben minden ügynök HTTP végpontot tenne közzé, publikálna egy Agent Card-ot, és az A2A JSON-RPC protokollon keresztül kommunikálna.


## Összefoglaló

Ebben a leckében megtanultad:

1. **Mi az A2A protokoll** — egy nyílt szabvány az ügynökök közötti felfedezésre, üzenetküldésre,
   és feladatkezelésre.
2. **Hogyan hozzunk létre specializált ügynököket** — egy valutaváltó ügynököt, egy tevékenységtervező ügynököt,
   és egy utazáskezelő koordinátort.
3. **Hogyan kapcsoljuk össze az ügynököket egy munkafolyamatba** — a `WorkflowBuilder` használatával az egymást követő
   üzenetküldés modellezésére az ügynökök között.
4. **Hogyan működik az A2A éles környezetben** — lehetővé téve a keretrendszerek és szolgáltatások közötti együttműködést
   dinamikus felfedezéssel és folyamatos frissítésekkel.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Felelősségkizárás:
Ez a dokumentum a Co-op Translator (https://github.com/Azure/co-op-translator) nevű mesterséges intelligencia alapú fordító szolgáltatással készült. Bár törekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. A dokumentum eredeti, anyanyelvi változatát kell tekinteni a hiteles forrásnak. Fontos információk esetén professzionális, emberi fordítást javaslunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
